# Assignment 3: Quantum Tunneling and Scattering

**Student Name:** [TYPE YOUR NAME HERE]  
**Date:** [TYPE DATE HERE]

### Objectives
1. Use the iterative "step" method (finite difference) to simulate a free particle hitting a potential energy barrier.
2. Understand plane waves, reflection, and transmission in quantum scattering.
3. Calculate the Transmission Coefficient ($T$).
4. Analyze how barrier width, barrier height, and particle mass exponentially alter the probability of tunneling.

In [ ]:
# --- RUN THIS CELL FIRST ---
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({
    'font.size': 11,
    'axes.linewidth': 1.2,
    'lines.linewidth': 2.0
})
print("Environment ready!")

## Part 1: Scattering and the Backwards Integration Trick

When a particle is free (not trapped in a well), it travels as a complex plane wave: $\psi(x) = A e^{ikx}$, where $k = \sqrt{2mE}/\hbar$ is the wave vector. 

When a plane wave hits a barrier from the left, it splits:
1. **Incident Wave:** Travels right, towards the barrier.
2. **Reflected Wave:** Bounces left, off the barrier.
3. **Transmitted Wave:** Tunnels through and travels right.

**The Computational Hack:** 
If we integrate from left to right, we have to guess the exact ratio of incident and reflected waves. Instead, we will start on the **right side** of the barrier, assuming a perfect transmitted wave $\psi(x) = e^{ikx}$ has already made it through. Then, we use our finite difference loop to integrate *backwards* (right-to-left) through the barrier to find out what incident wave on the left was required to create it!

From the Schrödinger equation, our iterative step is:
$$ \psi_{j-1} = 2\psi_j - \psi_{j+1} - \frac{2m (\Delta x)^2}{\hbar^2} (E - V_j) \psi_j $$
*(Note: Because wavefunctions use complex numbers, we must tell numpy our array is `dtype=complex`)*

In [ ]:
# Grid Parameters (from x = -5 to x = 5)
N = 1000
x = np.linspace(-5, 5, N)
dx = x[1] - x[0]

# Physical Parameters (Atomic Units: hbar = 1)
m = 1.0     # Mass of particle
E = 2.0     # Energy of incoming particle
V0 = 5.0    # Barrier height (Notice E < V0, classically forbidden!)
barrier_width = 1.0

# 1. Build the Rectangular Barrier
V = np.zeros(N)
for i in range(N):
    if abs(x[i]) <= barrier_width / 2.0:
        V[i] = V0

# 2. Initialize the Complex Wavefunction Array
psi = np.zeros(N, dtype=complex)

# Wave vector outside the barrier (where V=0)
k = np.sqrt(2.0 * m * E) 

# 3. Set Initial Conditions on the RIGHT side of the grid (Transmitted Wave)
psi[-1] = np.exp(1j * k * x[-1])
psi[-2] = np.exp(1j * k * x[-2])

# 4. Integrate BACKWARDS (from right to left)
### YOUR CODE HERE: Write a loop to calculate psi[j-1] ###
# Hint: Loop j backwards from N-2 down to 1.
# for j in range(N-2, 0, -1):
#     psi[j-1] = ...




print("Backwards integration complete!")

## Part 2: Visualizing the Wave and Calculating Transmission

Because we started with an amplitude of 1 on the right side ($|A_{\text{trans}}|^2 = 1$), the amplitude on the far left side of our grid now represents a mixture of the Incident and Reflected waves. 

To find the **Transmission Coefficient ($T$)**, we calculate the ratio of the transmitted probability density to the incident probability density. 

Let's visualize the real part of the wavefunction to see the amplitude decay as it crosses the barrier.

In [ ]:
# To find the incident amplitude, we extract it from the far left of the grid
# (Using a standard computational extraction for the incident component A)
A_inc = 0.5 * (psi[0] + (1j / k) * (psi[1] - psi[0]) / dx) * np.exp(-1j * k * x[0])

# Transmission probability T = |A_trans|^2 / |A_inc|^2
# Since we forced A_trans = 1, T is simply 1 / |A_inc|^2
T = 1.0 / np.abs(A_inc)**2
print(f"Transmission Coefficient (T): {T:.3e} (or {T * 100:.4f}%)")

# --- PLOTTING ---
plt.figure(figsize=(10, 5))
# Scale the potential down slightly just for visual comparison with the wave
plt.plot(x, V / V0, 'k--', label='Barrier (Scaled)') 
plt.plot(x, np.real(psi), 'b-', label='Re($\psi$)')

plt.title(f"Tunneling Wavefunction (Energy = {E}, Barrier = {V0})")
plt.xlabel("Position")
plt.ylabel("Amplitude")
plt.legend(loc='upper right')
plt.show()

## Part 3: The Parameters of Tunneling (Mass, Width, Height)

Tunneling is highly sensitive to the physical parameters of the system. Let's wrap our solver into a Python function so we can rapidly test different chemical scenarios.

### Your Task
Run the cell below to define the `calculate_transmission` function. Then, use the following code cells to see how $T$ changes when you alter:
1. The **Width** of the barrier.
2. The **Height** of the barrier ($V_0$).
3. The **Mass** of the particle (e.g., Hydrogen vs. Deuterium).

In [ ]:
# --- DO NOT MODIFY THIS CELL ---
def calculate_transmission(m, E, V0, width):
    """Calculates transmission probability T for given parameters."""
    k = np.sqrt(2.0 * m * E)
    V = np.zeros(N)
    for i in range(N):
        if abs(x[i]) <= width / 2.0:
            V[i] = V0
            
    psi_temp = np.zeros(N, dtype=complex)
    psi_temp[-1] = np.exp(1j * k * x[-1])
    psi_temp[-2] = np.exp(1j * k * x[-2])
    
    for j in range(N-2, 0, -1):
        psi_temp[j-1] = 2*psi_temp[j] - psi_temp[j+1] - (2*m*dx**2)*(E - V[j])*psi_temp[j]
        
    A_inc = 0.5 * (psi_temp[0] + (1j / k) * (psi_temp[1] - psi_temp[0]) / dx) * np.exp(-1j * k * x[0])
    return 1.0 / np.abs(A_inc)**2

In [ ]:
### YOUR CODE HERE: Experiment with the function ###

# Baseline: m=1.0, E=2.0, V0=5.0, width=1.0
T_baseline = calculate_transmission(1.0, 2.0, 5.0, 1.0)
print(f"Baseline T:  {T_baseline:.3e}")

# 1. Double the Width (width = 2.0)
# T_wide = calculate_transmission(...)
# print(f"Wider Barrier T: {T_wide:.3e}")

# 2. Double the Barrier Height (V0 = 10.0)
# T_tall = calculate_transmission(...)
# print(f"Taller Barrier T: {T_tall:.3e}")

# 3. Double the Mass (Isotope effect: H -> D) (m = 2.0)
# T_heavy = calculate_transmission(...)
# print(f"Heavier Mass T:  {T_heavy:.3e}")

### Conceptual Questions

**Question 1:** Based on your calculations in Part 3, does tunneling probability decay *linearly* or *exponentially* as barrier width increases? Explain how the shape of the wavefunction inside the barrier supports your answer.

**[DOUBLE CLICK THIS CELL TO TYPE YOUR ANSWER]**

**Question 2:** **The Kinetic Isotope Effect.** In organic chemistry, enzymes sometimes rely on quantum tunneling to move a Hydrogen atom (a proton, $m \approx 1$) across a barrier to catalyze a reaction. If a chemist mutates the substrate by replacing that Hydrogen with Deuterium ($m \approx 2$), the reaction rate slows down massively. Based on your "Heavier Mass" calculation, explain why the enzyme can easily move Hydrogen but struggles to move Deuterium.

**[DOUBLE CLICK THIS CELL TO TYPE YOUR ANSWER]**